# Random Forest: Predicting `made_it_contract` from Production & Athleticism Scores

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    classification_report,
    average_precision_score,
    PrecisionRecallDisplay,
    ConfusionMatrixDisplay,
)

RANDOM_STATE = 42

## 1. Load & Prepare Data

In [ ]:
df = pd.read_csv("../data/processed/draft_enriched_with_contracts.csv")

FEATURES = ["production_score", "athleticism_score"]
TARGET = "made_it_contract"

# Keep only rows with both feature scores and a valid target
model_df = df.dropna(subset=FEATURES + [TARGET]).copy()

# Coerce target to int (True/False -> 1/0)
model_df[TARGET] = model_df[TARGET].astype(int)

print(f"Total rows with complete data: {len(model_df)}")
print(f"\nTarget distribution:")
print(model_df[TARGET].value_counts())
print(f"\nPositive rate: {model_df[TARGET].mean():.1%}")
model_df[FEATURES].describe()

## 2. Train / Test Split

In [ ]:
X = model_df[FEATURES]
y = model_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Train size: {len(X_train)}  |  Test size: {len(X_test)}")

## 3. Train Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

rf.fit(X_train, y_train)
print("Model trained.")

## 4. Evaluation

In [ ]:
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

# PR-AUC (Average Precision) is the right metric when we only care about
# identifying players who made it. A random classifier scores = positive rate.
baseline = y_test.mean()
pr_auc = average_precision_score(y_test, y_prob)

print(classification_report(y_test, y_pred, target_names=["did not make it", "made it"]))
print(f"PR-AUC (Average Precision): {pr_auc:.4f}  |  Random baseline: {baseline:.4f}")

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(rf, X, y, cv=cv, scoring="average_precision")
print(f"5-fold CV PR-AUC: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")
print(f"Fold scores: {cv_scores.round(4)}")

## 5. Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=["Did Not Make It", "Made It"],
    ax=axes[0],
    colorbar=False,
)
axes[0].set_title("Confusion Matrix")

# Precision-Recall curve (positive class only)
PrecisionRecallDisplay.from_predictions(y_test, y_prob, ax=axes[1], name="Random Forest")
axes[1].axhline(y=y_test.mean(), color="k", linestyle="--", label=f"Random baseline ({y_test.mean():.2f})")
axes[1].set_title("Precision-Recall Curve (Made It)")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()

fig, ax = plt.subplots(figsize=(6, 3))
importances.plot.barh(ax=ax)
ax.set_xlabel("Mean Decrease in Impurity")
ax.set_title("Feature Importances")
plt.tight_layout()
plt.show()

print(importances)

In [ ]:
# Decision boundary plot over the full feature space
h = 0.5
x_min, x_max = X[FEATURES[0]].min() - 1, X[FEATURES[0]].max() + 1
y_min, y_max = X[FEATURES[1]].min() - 1, X[FEATURES[1]].max() + 1

xx, yy = np.meshgrid(
    np.arange(x_min, x_max, h),
    np.arange(y_min, y_max, h),
)
Z = rf.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1]
Z = Z.reshape(xx.shape)

fig, ax = plt.subplots(figsize=(8, 6))
contour = ax.contourf(xx, yy, Z, levels=20, cmap="RdYlGn", alpha=0.7)
plt.colorbar(contour, ax=ax, label="P(made_it)")

scatter = ax.scatter(
    X[FEATURES[0]], X[FEATURES[1]],
    c=y, cmap="bwr", edgecolors="k", linewidths=0.3, s=20, alpha=0.6,
)
ax.set_xlabel("production_score")
ax.set_ylabel("athleticism_score")
ax.set_title("RF Decision Boundary: P(made_it_contract)")
plt.tight_layout()
plt.show()